# CyTOF-transform: Quickstart

This notebook walks through the core features of the `cytof_transform` package:

- Generating synthetic CyTOF data with realistic technical variation
- Running QC checks on the technical factor and marker intensity regimes
- Applying global CyTOF-transform normalisation (sctransform-style)
- Multi-batch correction via balanced sampling with `line_col`
- Compartment-aware normalisation with `cytof_transform_by_compartment`
- Accessing corrected intensities and z-scored residuals for downstream analysis

## 1. Installation

Install the package from PyPI:

```bash
pip install cytof-transform
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cytof_transform import (
    CytofTransformConfig,
    cytof_transform_global,
    cytof_transform_by_compartment,
    compute_marker_tech_correlations,
    evaluate_marker_intensity_regime,
    plot_tech_factor_qc,
    plot_marker_correlations_qc,
    plot_gamma_qc,
)

## 2. Synthetic data

We generate a synthetic CyTOF dataset with 2 000 cells.
A log-normal technical factor is shared across all channels, mimicking cell-size /
staining-efficiency variation.  Control markers (histones and DNA intercalators)
are driven almost entirely by the technical factor, while biological markers carry
both a biological signal and a weaker technical contamination.  Everything is
arcsinh-transformed, as is standard for CyTOF data.

In [ ]:
np.random.seed(42)
n_cells = 2000

# --- Technical factor (log-normal, median-normalised to 1) ---
tech = np.random.lognormal(mean=0, sigma=0.6, size=n_cells)
tech = tech / np.median(tech)

# --- Control markers (histones + DNA intercalators) ---
control_marker_names = ["H3", "H4", "H2A", "H2B", "DNA1", "DNA2"]
gamma_ctrl_values   = [1.2, 1.1, 0.9, 1.0, 0.8, 0.85]

control_data = {}
for name, gamma_ctrl in zip(control_marker_names, gamma_ctrl_values):
    noise = np.random.normal(0, 0.3, n_cells)
    raw = np.log(tech) * gamma_ctrl + noise
    control_data[name] = np.arcsinh(raw)

# --- Biological markers ---
bio_marker_names = ["CD3", "CD4", "CD8a", "CD20", "CD45", "Ki67", "pS6", "pERK"]
bio_mus          = [1.0,   0.5,   0.5,    0.8,    2.0,    1.2,    0.3,   0.3]
gamma_bio_values = [0.6,   0.4,   0.5,    0.3,    0.7,    0.8,    0.2,   0.15]

bio_data = {}
for name, mu, gamma_bio in zip(bio_marker_names, bio_mus, gamma_bio_values):
    biology = np.random.normal(mu, 0.8, n_cells)
    raw = biology + np.log(tech) * gamma_bio
    bio_data[name] = np.arcsinh(raw)

# --- Metadata ---
sample_ids  = ["S1", "S2", "S3", "S4"]
sample_col  = np.array([sample_ids[i % 4] for i in range(n_cells)])

compartment_col = np.empty(n_cells, dtype=object)
compartment_col[:800]       = "immune"
compartment_col[800:1500]   = "tumour"
compartment_col[1500:]      = "stroma"

# --- Assemble DataFrame ---
asinh_data = pd.DataFrame({**control_data, **bio_data})
asinh_data["sample_id"]   = sample_col
asinh_data["compartment"] = compartment_col

print("Shape:", asinh_data.shape)
asinh_data.head()

## 3. Technical factor QC

`plot_tech_factor_qc` visualises the correlation structure of the control markers
and the distribution of the inferred technical factor.  Tight, consistent
correlations among the control channels are a sign that the technical factor
is well-identified.

In [ ]:
control_markers = ["H3", "H4", "H2A", "H2B", "DNA1", "DNA2"]

plot_tech_factor_qc(asinh_data[control_markers], control_markers)

## 4. Evaluate marker intensity regime

`evaluate_marker_intensity_regime` flags whether each candidate biological marker
appears to be predominantly zero / near-zero (sparse regime) or has a meaningful
positive signal.  The result informs which markers are safe to include in the
gamma-regression step.

In [ ]:
bio_markers = ["CD3", "CD4", "CD8a", "CD20", "CD45", "Ki67", "pS6", "pERK"]

regime_df = evaluate_marker_intensity_regime(asinh_data, candidate_markers=bio_markers)
print(regime_df)

## 5. Global CyTOF-transform

`cytof_transform_global` fits a per-marker gamma (technical loading) by regressing
each biological marker against the technical factor estimated from the control
channels.  The corrected values are the arcsinh-scale residuals after removing the
technical component.  Setting `anchor_to_median=True` shifts the corrected values
back to the sample median, preserving interpretable arcsinh intensities.  Setting
`zscore=True` additionally stores z-scored residuals in `result.residuals_z`,
which are ideal for dimensionality-reduction methods such as PCA and UMAP.

In [ ]:
control_markers = ["H3", "H4", "H2A", "H2B", "DNA1", "DNA2"]
bio_markers = ["CD3", "CD4", "CD8a", "CD20", "CD45", "Ki67", "pS6", "pERK"]

config = CytofTransformConfig(
    control_markers=control_markers,
    markers_to_correct=bio_markers,
    anchor_to_median=True,
    zscore=True,
)
result = cytof_transform_global(asinh_data, config)
print("gamma values:")
for m, g in result.gamma.items():
    print(f"  {m}: {g:.3f}")

## 6. QC: marker correlations before vs after

`plot_marker_correlations_qc` compares the Pearson correlation of each biological
marker with the technical factor before and after correction.  Successful
normalisation should drive those correlations close to zero.

In [ ]:
plot_marker_correlations_qc(
    asinh_data[bio_markers],
    result.corrected[bio_markers],
    result.tech_factor,
)

## 7. QC: gamma values

`plot_gamma_qc` plots the fitted gamma for each biological marker, making it easy
to spot markers with unexpectedly high or low technical loadings.

In [ ]:
plot_gamma_qc(result.gamma)

## 8. Multi-batch correction with balanced sampling

When the dataset spans multiple acquisition batches or samples, the technical factor
estimation can be biased if one batch dominates.  Passing `line_col="sample_id"` to
`CytofTransformConfig` instructs `cytof_transform_global` to draw a balanced
subsample from each unique value of that column before fitting the gamma regression,
ensuring no single batch drives the model.

In [ ]:
config_balanced = CytofTransformConfig(
    control_markers=control_markers,
    markers_to_correct=bio_markers,
    anchor_to_median=True,
    zscore=True,
    line_col="sample_id",
)
result_balanced = cytof_transform_global(asinh_data, config_balanced)
print("Done. gamma (balanced):")
for m, g in result_balanced.gamma.items():
    print(f"  {m}: {g:.3f}")

## 9. Compartment-aware normalisation

In tumour microenvironment studies, distinct tissue compartments (immune infiltrate,
tumour cells, stroma) can have very different baseline intensities.  A single global
gamma may be suboptimal.  `cytof_transform_by_compartment` fits a separate model
for each compartment and returns a combined result with per-compartment gamma
estimates averaged for reporting.

In [ ]:
compartments = asinh_data["compartment"]
result_comp = cytof_transform_by_compartment(
    asinh_data=asinh_data,
    compartments=compartments,
    config=config,
)
print("Compartment-aware correction done.")
print("Mean gamma across compartments:")
for m, g in result_comp.gamma.items():
    print(f"  {m}: {g:.3f}")

## 10. Using the corrected data

The `CytofTransformResult` object exposes two corrected representations:

- **`result.corrected`** — corrected arcsinh-scale intensities.  Use these when
  you need interpretable values (e.g. gating, heatmaps, violin plots).
- **`result.residuals_z`** — z-scored residuals after technical-factor removal.
  These are variance-stabilised and mean-zero, making them the recommended input
  for PCA, UMAP, clustering, and other dimensionality-reduction workflows.

Both are `pd.DataFrame` objects with the same index as the input data, so they
can be concatenated with metadata columns directly.

In [ ]:
# corrected arcsinh values — interpretable intensities
corrected = result.corrected[bio_markers]

# z-scored residuals — use for PCA / UMAP
residuals_z = result.residuals_z[bio_markers]

print("Corrected (arcsinh), first 3 rows:")
print(corrected.head(3).round(3))
print("\nZ-scored residuals, first 3 rows:")
print(residuals_z.head(3).round(3))

## Summary

This quickstart demonstrated the main `cytof_transform` workflows:

- **Technical factor QC** — `plot_tech_factor_qc` to inspect control-marker
  correlations and the inferred technical factor distribution.
- **Intensity regime evaluation** — `evaluate_marker_intensity_regime` to flag
  sparse or near-zero markers before fitting.
- **Global normalisation** — `cytof_transform_global` with
  `CytofTransformConfig` to fit per-marker gammas and produce corrected values.
- **Post-correction QC** — `plot_marker_correlations_qc` and `plot_gamma_qc`
  to verify that technical correlations have been removed and that gamma estimates
  are sensible.
- **Multi-batch balanced fitting** — `line_col` argument to prevent batch-size
  imbalance from dominating the gamma regression.
- **Compartment-aware correction** — `cytof_transform_by_compartment` for
  datasets with distinct tissue compartments.
- **Downstream-ready outputs** — `result.corrected` for interpretable intensities
  and `result.residuals_z` for PCA / UMAP.